In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import multiprocessing as mp
from structure_SYNCHRONOUS_FNs_singlesideBASEreinLEARNING2_2026 import play_sequence

np.set_printoptions(suppress=True)

In [ ]:
# use this cell for setting initial parameters

sides = 2
terms = 7
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps*1


def make_linear_pairs_7term():
    """Generate pair arrays for 7-term transitive inference.
    
    Adjacent pairs (distance 1) for training.
    Distance 3+ non-adjacent pairs for cross-set training (side 1 in 2-side mode).
    Distance 2 pairs held out as test set to measure symbolic distance effect.
    """
    n = terms
    
    # adjacent pairs (distance 1)
    adj = []
    for i in range(n - 1):
        adj.append([i, i+1])
        adj.append([i+1, i])
    pairs = np.asarray(adj, dtype=np.int64)
    
    # non-adjacent pairs at distance >= 3 (for training in 2-side mode)
    nonadj = []
    for i in range(n):
        for j in range(i + 3, n):
            nonadj.append([i, j])
            nonadj.append([j, i])
    nonadjpairs = np.asarray(nonadj, dtype=np.int64)
    
    # test pairs: distance 2 pairs held out
    test = []
    for i in range(n - 2):
        test.append([i, i+2])
        test.append([i+2, i])
    testpairs = np.asarray(test, dtype=np.int64)
    
    # all pairs = adjacent + non-adjacent + test
    allpairs = np.concatenate((pairs, nonadjpairs, testpairs))
    
    # reward arrays: higher index is rewarded (standard linear TI rule)
    pairs_reward = np.asarray([[1,0] if a > b else [0,1] for a,b in pairs], dtype=np.int64)
    nonadjpairs_reward = np.asarray([[1,0] if a > b else [0,1] for a,b in nonadjpairs], dtype=np.int64)
    testpairs_reward = np.asarray([[1,0] if a > b else [0,1] for a,b in testpairs], dtype=np.int64)
    allpairs_reward = np.asarray([[1,0] if a > b else [0,1] for a,b in allpairs], dtype=np.int64)
    
    return pairs, testpairs, nonadjpairs, allpairs, pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward


pairs, testpairs, nonadjpairs, allpairs, pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward = make_linear_pairs_7term()

plen = len(pairs)
alen = len(allpairs)
ntest = len(testpairs)

print(f"terms = {terms}")
print(f"sides = {sides}")
print(f"adjacent pairs: {plen}")
print(f"non-adjacent pairs: {len(nonadjpairs)}")
print(f"test pairs: {ntest}")
print(f"all pairs: {alen}")
print(f"test pair indices: {testpairs.tolist()}")

In [ ]:
# use this cell for running the code

# measuring how long code takes to run
start = time.perf_counter()


pool = mp.Pool(processes=3)

sg = SeedSequence()
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]


mpseq_final = pool.starmap(play_sequence, [(n, rgs[n], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward, plen, alen, terms, maxvalue, startstop, noise, annealing, runs, inertia, blocklength) for n in range(runs)])

final_sigweights = []
final_cumsuc = []
final_cumsucadd = []
final_testcumsucadd = []
final_recweights = []
final_test_pair_stats = []

for res_idx in range(0, len(mpseq_final)):
    
    final_sigweights.append(mpseq_final[res_idx][0])
    final_cumsuc.append(mpseq_final[res_idx][1])
    final_cumsucadd.append(mpseq_final[res_idx][2])
    final_testcumsucadd.append(mpseq_final[res_idx][3])
    final_recweights.append(mpseq_final[res_idx][4])
    final_test_pair_stats.append(mpseq_final[res_idx][5])

final_sigweights = np.asarray(final_sigweights)
final_cumsuc = np.asarray(final_cumsuc)
final_cumsucadd = np.asarray(final_cumsucadd)
final_testcumsucadd = np.asarray(final_testcumsucadd)
final_recweights = np.asarray(final_recweights)
final_test_pair_stats = np.asarray(final_test_pair_stats)


print("average cumsuc = ")
print(np.average(final_cumsuc)/runs)
print(" ")

print("average final cumsucadd = ")
print(np.average(final_cumsucadd)/(100))
print(" ")

print("average test cumsum = ")
print(np.average(final_testcumsucadd)/(100))
print(" ")

finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

In [ ]:
np.save('structure_SYNCHRONOUS_7s_2s_BASElearning-MV10_sigweights', final_sigweights)
np.save('structure_SYNCHRONOUS_7s_2s_BASElearning-MV10_recweights', final_recweights)
np.save('structure_SYNCHRONOUS_7s_2s_BASElearning-MV10_testcumsucadd', final_testcumsucadd)
np.save('structure_SYNCHRONOUS_7s_2s_BASElearning-MV10_test_pair_stats', final_test_pair_stats)

In [ ]:
cutoff = 90
final_test_count = 0
for cumsum in final_testcumsucadd:
    if cumsum > cutoff:
        final_test_count += 1
print(f"Runs with test accuracy > {cutoff}/100: {final_test_count}")

In [ ]:
# Per-pair distance effect analysis

# distances for each test pair
test_distances = np.abs(testpairs[:, 0] - testpairs[:, 1])

# shape: (runs, n_test_pairs, 2) where col 0 = attempts, col 1 = successes
print(f"test_pair_stats shape: {final_test_pair_stats.shape}")
print(f"Number of runs: {final_test_pair_stats.shape[0]}")
print(f"Number of test pairs: {final_test_pair_stats.shape[1]}")

print("\n--- Per-pair test results (all runs) ---")
print(f"{'Pair':<12} {'Dist':<6} {'Attempts':<10} {'Successes':<10} {'Accuracy':<10}")
print("-" * 50)

for i in range(ntest):
    total_attempts = np.sum(final_test_pair_stats[:, i, 0])
    total_successes = np.sum(final_test_pair_stats[:, i, 1])
    accuracy = total_successes / total_attempts if total_attempts > 0 else 0
    dist = test_distances[i]
    pair_str = f"({testpairs[i][0]}, {testpairs[i][1]})"
    print(f"{pair_str:<12} {dist:<6} {total_attempts:<10} {total_successes:<10} {accuracy:<10.4f}")

print("\n--- Aggregated by distance ---")
unique_dists = np.unique(test_distances)
print(f"{'Distance':<10} {'Pairs':<8} {'Attempts':<12} {'Successes':<12} {'Accuracy':<10} {'StdErr':<10}")
print("-" * 65)

for d in sorted(unique_dists):
    mask = test_distances == d
    pair_indices = np.where(mask)[0]
    
    total_attempts = np.sum(final_test_pair_stats[:, pair_indices, 0])
    total_successes = np.sum(final_test_pair_stats[:, pair_indices, 1])
    overall_accuracy = total_successes / total_attempts if total_attempts > 0 else 0
    
    # per-run accuracy for standard error
    run_accuracies = []
    for r in range(runs):
        r_attempts = np.sum(final_test_pair_stats[r, pair_indices, 0])
        r_successes = np.sum(final_test_pair_stats[r, pair_indices, 1])
        if r_attempts > 0:
            run_accuracies.append(r_successes / r_attempts)
    
    n_pairs_at_dist = len(pair_indices)
    if len(run_accuracies) > 0:
        std_err = np.std(run_accuracies) / np.sqrt(len(run_accuracies))
    else:
        std_err = 0
    
    print(f"{d:<10} {n_pairs_at_dist:<8} {total_attempts:<12} {total_successes:<12} {overall_accuracy:<10.4f} {std_err:<10.4f}")

In [ ]:
# Summary: mean accuracy by distance across all runs
print("Distance effect summary (mean accuracy across runs by distance):")
print(f"{'Distance':<10} {'Mean Accuracy':<15} {'Std Err':<10}")
print("-" * 38)

for d in sorted(unique_dists):
    mask = test_distances == d
    pair_indices = np.where(mask)[0]
    
    run_accuracies = []
    for r in range(runs):
        r_attempts = np.sum(final_test_pair_stats[r, pair_indices, 0])
        r_successes = np.sum(final_test_pair_stats[r, pair_indices, 1])
        if r_attempts > 0:
            run_accuracies.append(r_successes / r_attempts)
    
    if len(run_accuracies) > 0:
        mean_acc = np.mean(run_accuracies)
        std_err = np.std(run_accuracies) / np.sqrt(len(run_accuracies))
        print(f"{d:<10} {mean_acc:<15.4f} {std_err:<10.4f}")

In [ ]:
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup

@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype = numba.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000] 
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1] 
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count

In [ ]:
runs_dup_bins(runs, final_sigweights, terms)